# Semester 3 — Agent Engineering: Práctica

**Objetivo:** Construir un agente desde cero sin frameworks: bucle ReAct, herramientas, verifier, self-correction y memoria de sesión con compresión.

Sin LangChain, sin LangGraph. Código mínimo que expone exactamente qué hace un agente en cada paso.

**Prerequisito:** Variable de entorno `OPENAI_API_KEY` configurada.

## Setup

In [ ]:
!pip install openai --quiet

## Ejercicio 1 — Bucle ReAct desde cero

**Qué practicamos:** Implementar el bucle Reasoning → Action → Observation sin ningún framework. El LLM razona en texto libre, el código extrae la acción, ejecuta la herramienta y devuelve la observación.

**Por qué importa arquitectónicamente:** Entender el bucle ReAct sin abstracción te permite depurar cualquier agente. Cuando un agente basado en framework falla, si no conoces el bucle subyacente no puedes encontrar el problema. Además, muchos sistemas de producción no necesitan un framework completo — solo este bucle con 2-3 herramientas.

In [ ]:
import os
import re
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Herramienta mock: calculadora
def calculator(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

TOOLS_DESCRIPTION = """
Tienes acceso a una herramienta:
- calculator(expression): Evalúa expresiones matemáticas. Ej: calculator(2 + 3 * 4)

Para usar una herramienta, responde EXACTAMENTE así:
THOUGHT: <tu razonamiento>
ACTION: calculator(<expresión>)

Si ya tienes la respuesta final, responde:
THOUGHT: <razonamiento final>
ANSWER: <respuesta>
"""

def react_loop(question: str, max_steps: int = 5) -> str:
    messages = [
        {"role": "system", "content": TOOLS_DESCRIPTION},
        {"role": "user",   "content": question}
    ]

    for step in range(max_steps):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0
        )
        response_text = r.choices[0].message.content
        messages.append({"role": "assistant", "content": response_text})

        print(f"--- Paso {step + 1} ---")
        print(response_text)

        # ¿Respuesta final?
        if "ANSWER:" in response_text:
            answer_match = re.search(r"ANSWER:\s*(.+)", response_text, re.DOTALL)
            return answer_match.group(1).strip() if answer_match else response_text

        # ¿Acción a ejecutar?
        action_match = re.search(r"ACTION:\s*calculator\((.+?)\)", response_text)
        if action_match:
            expr = action_match.group(1)
            observation = calculator(expr)
            print(f"OBSERVATION: calculator({expr}) = {observation}\n")
            messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})

    return "Max steps reached without answer"

answer = react_loop("Si tengo 3 cajas con 17 manzanas cada una y vendo el 40%, ¿cuántas me quedan?")
print(f"\n=== RESPUESTA FINAL: {answer} ===")

## Ejercicio 2 — Añadir herramienta de búsqueda mock

**Qué practicamos:** Extender el agente ReAct con una segunda herramienta de búsqueda simulada. El agente decide qué herramienta usar según el contexto.

**Por qué importa arquitectónicamente:** El diseño del espacio de herramientas es una decisión arquitectónica crítica. Demasiadas herramientas confunden al LLM; muy pocas limitan su capacidad. Las descripciones de herramientas son código — un mal nombre de herramienta genera el mismo tipo de bugs que una función mal nombrada.

In [ ]:
import os
import re
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Base de conocimiento mock
KNOWLEDGE_BASE = {
    "python": "Python es un lenguaje de programación interpretado creado en 1991 por Guido van Rossum.",
    "openai": "OpenAI es una empresa de IA fundada en 2015. Creó GPT-4 y ChatGPT.",
    "llm": "Un LLM (Large Language Model) es un modelo de ML entrenado en grandes corpus de texto.",
    "token": "Un token es la unidad mínima de procesamiento de un LLM. Aprox 0.75 palabras en inglés.",
}

def search(query: str) -> str:
    query_lower = query.lower()
    for key, value in KNOWLEDGE_BASE.items():
        if key in query_lower:
            return value
    return "No encontré información sobre ese tema."

def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

TOOLS_DESCRIPTION = """
Herramientas disponibles:
- search(query): Busca información en la base de conocimiento. Ej: search(python)
- calculator(expression): Evalúa expresiones matemáticas. Ej: calculator(2 + 2)

Formato:
THOUGHT: <razonamiento>
ACTION: <tool>(<args>)

O si tienes la respuesta:
THOUGHT: <razonamiento>
ANSWER: <respuesta>
"""

def dispatch(action_text: str) -> str | None:
    if m := re.search(r"search\((.+?)\)", action_text):
        return search(m.group(1))
    if m := re.search(r"calculator\((.+?)\)", action_text):
        return calculator(m.group(1))
    return None

def agent(question: str, max_steps: int = 6) -> str:
    messages = [
        {"role": "system", "content": TOOLS_DESCRIPTION},
        {"role": "user",   "content": question}
    ]
    for step in range(max_steps):
        r = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages, temperature=0
        )
        text = r.choices[0].message.content
        messages.append({"role": "assistant", "content": text})
        print(f"[paso {step+1}] {text.strip()[:120]}")

        if "ANSWER:" in text:
            m = re.search(r"ANSWER:\s*(.+)", text, re.DOTALL)
            return m.group(1).strip() if m else text

        obs = dispatch(text)
        if obs:
            messages.append({"role": "user", "content": f"OBSERVATION: {obs}"})
    return "timeout"

print("=== Pregunta mixta ===")
result = agent("¿En qué año fue fundada OpenAI? ¿Cuántos años han pasado desde entonces hasta 2025?")
print(f"\nRESPUESTA: {result}")

## Ejercicio 3 — Verifier: rechazar outputs que no cumplen criterios

**Qué practicamos:** Implementar una capa de verificación que valida el output del agente contra criterios definidos antes de devolver la respuesta al usuario.

**Por qué importa arquitectónicamente:** El verifier es la diferencia entre un agente experimental y uno de producción. Sin verificación externa, confías ciegamente en que el LLM produce outputs correctos. Con verificación, tienes un guardarraíl que puede rechazar, pedir reintento, o escalar a un humano.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def generate_response(question: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0.7  # algo de variabilidad para ver diferentes casos
    )
    return r.choices[0].message.content

def verify(question: str, response: str, criteria: list[str]) -> dict:
    """Verifica si la respuesta cumple todos los criterios. Devuelve {passed, failures}."""
    criteria_str = "\n".join(f"- {c}" for c in criteria)
    prompt = f"""Evalúa si la respuesta cumple TODOS los criterios. Responde con JSON.

Pregunta: {question}
Respuesta: {response}
Criterios:
{criteria_str}

JSON: {{"passed": true/false, "failures": ["criterio fallado 1", ...]}}"""

    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    return json.loads(r.choices[0].message.content)

def verified_agent(question: str, criteria: list[str], max_retries: int = 3) -> str:
    """Genera respuesta y la reintenta si no pasa el verifier."""
    for attempt in range(max_retries):
        response = generate_response(question)
        result = verify(question, response, criteria)

        print(f"Intento {attempt + 1}: passed={result['passed']}")
        if result.get("failures"):
            print(f"  Fallos: {result['failures']}")

        if result["passed"]:
            return response

    return "[VERIFICACIÓN FALLIDA TRAS MÁXIMOS REINTENTOS]"

# Caso de uso: respuestas de soporte que deben cumplir criterios de calidad
question = "¿Cómo puedo mejorar la velocidad de mi sitio web?"
criteria = [
    "Menciona al menos 3 técnicas específicas",
    "No usa lenguaje vago como 'optimizar' sin explicar cómo",
    "La respuesta tiene menos de 200 palabras",
]

final = verified_agent(question, criteria)
print(f"\nRespuesta final:\n{final}")

## Ejercicio 4 — Self-correction: el agente revisa su propio output

**Qué practicamos:** Implementar un ciclo donde el agente genera una respuesta, la revisa críticamente, y produce una versión mejorada.

**Por qué importa arquitectónicamente:** Self-correction mejora la calidad del output sin necesidad de un modelo más potente. Es especialmente útil en tareas de generación de código, análisis y escritura estructurada. El coste es 2-3x más tokens, pero puede sustituir un modelo más caro con uno más barato en ciclo.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def generate(task: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": task}],
        temperature=0.7
    )
    return r.choices[0].message.content

def critique(task: str, response: str) -> str:
    """Genera crítica del output sin proponer solución todavía."""
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": task},
            {"role": "assistant", "content": response},
            {"role": "user", "content": "Revisa tu respuesta anterior. ¿Qué está incompleto, incorrecto o podría mejorarse? Sé específico y crítico."}
        ],
        temperature=0
    )
    return r.choices[0].message.content

def revise(task: str, original: str, criticism: str) -> str:
    """Genera versión mejorada incorporando la crítica."""
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user",      "content": task},
            {"role": "assistant", "content": original},
            {"role": "user",      "content": f"Crítica recibida: {criticism}\n\nAhora escribe una versión mejorada que aborde esos problemas."}
        ],
        temperature=0
    )
    return r.choices[0].message.content

def self_correcting_agent(task: str) -> dict:
    v1 = generate(task)
    criticism = critique(task, v1)
    v2 = revise(task, v1, criticism)
    return {"v1": v1, "criticism": criticism, "v2": v2}

task = "Explica qué es un sistema multi-agente en IA en 3 puntos clave para un CTO no técnico."
result = self_correcting_agent(task)

print("=== VERSIÓN INICIAL ===")
print(result["v1"])
print("\n=== CRÍTICA ===")
print(result["criticism"])
print("\n=== VERSIÓN MEJORADA ===")
print(result["v2"])

## Ejercicio 5 — Memoria de sesión con historial comprimido

**Qué practicamos:** Mantener historial de conversación y comprimir automáticamente cuando supera un threshold de tokens.

**Por qué importa arquitectónicamente:** Una conversación sin compresión crece linealmente en tokens y coste. Una conversación larga puede superar la ventana de contexto. La compresión periódica mantiene el coste bajo control sin perder el hilo de la conversación — es el mismo patrón que usa la memoria episódica en agentes de producción.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

TOKEN_THRESHOLD = 500  # umbral bajo para demostrar compresión

def estimate_tokens(messages: list[dict]) -> int:
    """Estimación rápida: ~4 chars por token. ponytail: aproximación, usar tiktoken si la precisión importa."""
    total_chars = sum(len(m.get("content", "")) for m in messages)
    return total_chars // 4

def compress_history(messages: list[dict]) -> list[dict]:
    """Resume el historial en un mensaje de sistema comprimido."""
    history_text = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
        if m["role"] != "system"
    )
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Resume la siguiente conversación en máximo 100 palabras, preservando los puntos clave y decisiones tomadas."},
            {"role": "user",   "content": history_text}
        ],
        temperature=0
    )
    summary = r.choices[0].message.content
    return [{"role": "system", "content": f"[RESUMEN DE CONVERSACIÓN ANTERIOR]: {summary}"}]

class SessionMemory:
    def __init__(self, system_prompt: str):
        self.messages = [{"role": "system", "content": system_prompt}]
        self.compressions = 0

    def chat(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})

        tokens = estimate_tokens(self.messages)
        if tokens > TOKEN_THRESHOLD:
            print(f"  [COMPRIMIENDO historial: ~{tokens} tokens → compresión #{self.compressions + 1}]")
            self.messages = compress_history(self.messages)
            self.messages.append({"role": "user", "content": user_input})
            self.compressions += 1

        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=self.messages,
            temperature=0
        )
        response = r.choices[0].message.content
        self.messages.append({"role": "assistant", "content": response})
        return response

    def stats(self) -> dict:
        return {"messages": len(self.messages), "approx_tokens": estimate_tokens(self.messages), "compressions": self.compressions}

# Demo de conversación con compresión automática
memory = SessionMemory("Eres un experto en arquitectura de sistemas de IA. Responde de forma concisa.")

turns = [
    "¿Cuál es la diferencia entre un agente y un pipeline de IA?",
    "¿Cuándo usarías un pipeline en vez de un agente?",
    "¿Qué herramientas debería tener un agente de soporte al cliente?",
    "Volviendo a lo que dijiste sobre pipelines, ¿cómo se conectan con lo de las herramientas?",
    "¿Cuál es el mayor riesgo de producción que mencionaste antes?",
]

for user_input in turns:
    response = memory.chat(user_input)
    print(f"Usuario: {user_input}")
    print(f"Agente:  {response[:100]}...")
    print(f"Stats:   {memory.stats()}\n")